# Soda Core — Data Quality Tests

Run **`install_soda.ipynb`** first on the cluster.

Then run all cells: Soda scans for bronze, silver, gold → `ecom_clickstream.dq.*`

In [1]:
# DBTITLE 1,Setup
import sys
from pathlib import Path

for candidate in [Path("."), Path("soda"), Path("../soda")]:
    if (candidate / "dq_data.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise FileNotFoundError("Could not find soda/ — run from repo root layout or deploy bundle first")

from dq_data import find_soda_dir, load_datasets
from dq_run import _run_one_scan
from dq_persist import SUMMARY_TABLE, RESULTS_TABLE

SODA_DIR = find_soda_dir()
registry = load_datasets(SODA_DIR)

In [2]:
# DBTITLE 1,Run DQ tests
scan_id = _run_one_scan(spark, SODA_DIR, registry)
# Optional: layers=["silver", "gold"]

Tables ready: ecom_clickstream.dq.scan_summary, ecom_clickstream.dq.check_results

--- bronze / contracts ---
[missing_count(entity_type) = 0] PASS (check_value: 0)
[invalid_count(entity_type) = 0] PASS (check_value: 0)
[missing_count(payload) = 0] PASS (check_value: 0)
[failed rows] PASS (value: 0)

--- bronze / freshness_and_schema_drift ---
[missing_count(_ingested_at) = 0] PASS (check_value: 0)
[missing_count(_source_file) = 0] PASS (check_value: 0)

--- bronze / expectations ---
[failed rows] PASS (value: 0)
[failed rows] PASS (value: 0)

--- bronze / anomalies ---
[row_count between 700000 and 800000] PASS (check_value: 751501)
[row_count between 900000 and 1100000] PASS (check_value: 985696)
[row_count between 15000 and 25000] PASS (check_value: 21020)

--- bronze / integrity ---
[row_count > 0] PASS (check_value: 751501)
[row_count > 0] PASS (check_value: 985696)

--- silver / contracts ---
[row_count > 0] PASS (check_value: 726514)
[missing_count(user_id) = 0] PASS (check_valu

In [ ]:
# DBTITLE 1,Results
display(
    spark.table(SUMMARY_TABLE)
    .filter(f"scan_id = '{scan_id}'")
    .orderBy("medallion_layer", "check_category")
)
display(
    spark.table(RESULTS_TABLE)
    .filter(f"scan_id = '{scan_id}'")
    .orderBy("medallion_layer", "check_name")
)
display(
    spark.sql(f"""
        SELECT medallion_layer, check_category, dataset_name,
               check_name, outcome, check_value
        FROM {RESULTS_TABLE}
        WHERE scan_id = '{scan_id}'
          AND outcome IN ('fail', 'warn')
        ORDER BY outcome DESC, medallion_layer, check_category
    """)
)